In [ ]:
# ==============================================================================
# DATABASE OF RELIGIOUS HISTORY (DRH) - TAGGING TREES
# 
# ARCHITECTURE NOTE: The DRH is a React-based SPA that fetches its taxonomy via 
# a Hasura GraphQL API. This script executes a single POST request to the 
# GraphQL endpoint to fetch the raw relational JSON for the Tagging Trees.
# (Note: Poll variables are handled in Cell 2).
# 
# SSSOM ALIGNMENT STRATEGY: 
# The script targets 'Religious Group' and 'Religious Text'.
# Because the DRH is not LOD, URIs are left blank and primary keys are synthesized 
# using the DRH:[id] format.
#
# TEXT NORMALIZATION RULE: DRH strings may contain invisible Unicode Line Separators 
# (LS). The script aggressively flattens all \s+ into standard spaces.
#
# PATHING RULE:
# Deduplicates redundant root nodes returned by the API to prevent path stuttering 
# (e.g., preventing "Religious Group > Religious Group > ..."). Localized and 
# test branches are intentionally retained in the Bronze layer for future cleaning.
# ==============================================================================

import os
import sys
import requests
import json
import re
import pandas as pd
import time
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "DRH"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="Database of Religious History",
    fallback_uri="" 
)
SOURCE_NAME = registry_data['Source_Name']
# Updated file name for parity with the Polls script
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}_trees.csv")

GRAPHQL_URL = "https://religiondatabase.org/v1/graphql"
HEADERS = {
    'User-Agent': 'ReligiousMappingProject/1.0',
    'Content-Type': 'application/json',
    'Accept': 'application/json'
}

GRAPHQL_QUERY = """
query GetTaggingTrees {
  polls_entrytaggroup {
    id
    name
    tags {
      id
      name
      level
      parent_tag_id
    }
  }
}
"""

TARGET_GROUPS = [
    "Religious Group", 
    "Religious Text"
]

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
    else:
        print(f"[*] Found existing DRH trees file. Overwriting for fresh bulk GraphQL extraction...")
        os.remove(raw_ingest_file)

# --- 4. Text Normalization Helper ---
def clean_text(text):
    """Flattens all standard and unicode whitespace (including LS/PS) into a single space."""
    if not text: return ""
    return re.sub(r'\s+', ' ', str(text)).strip()

def get_absolute_path(node_id, nodes_dict, depth=0):
    """Recursively builds the breadcrumb path from the root node."""
    if depth > 20: 
        return nodes_dict.get(node_id, {}).get("name", "Unknown")
        
    node = nodes_dict.get(node_id)
    if not node: return "Unknown"
        
    parent_id = node.get('parent_tag_id')
    
    if not parent_id or parent_id not in nodes_dict:
        return clean_text(node['name'])
        
    parent_path = get_absolute_path(parent_id, nodes_dict, depth + 1)
    return f"{parent_path} > {clean_text(node['name'])}"

# --- 5. Main Extraction & API Call ---
print(f"[*] Executing GraphQL query against {GRAPHQL_URL}...")

payload = {"query": GRAPHQL_QUERY}
all_rows = []

max_retries = 3
for attempt in range(max_retries):
    try:
        response = requests.post(GRAPHQL_URL, headers=HEADERS, json=payload, timeout=20)
        response.raise_for_status()
        json_data = response.json()
        break
    except requests.exceptions.RequestException as e:
        print(f"[!] API Request failed (Attempt {attempt+1}/{max_retries}): {e}")
        if attempt == max_retries - 1:
            print("[!] CRITICAL ERROR: Maximum retries reached. Exiting.")
            sys.exit(1)
        time.sleep(2)

tag_groups = json_data.get('data', {}).get('polls_entrytaggroup', [])
print(f"[*] Successfully retrieved {len(tag_groups)} tagging trees from DRH.")

for group in tag_groups:
    group_name = clean_text(group.get('name'))
    
    if group_name not in TARGET_GROUPS:
        print(f"    [-] Skipping out-of-scope tree: '{group_name}'")
        continue
        
    tags = group.get('tags', [])
    print(f"    [+] Processing Target Tree: '{group_name}' ({len(tags)} nodes)...")
    
    nodes_dict = {tag['id']: tag for tag in tags}
    
    for node_id, node in nodes_dict.items():
        cid = str(node_id)
        name = clean_text(node.get('name'))
        
        parent_id_raw = node.get('parent_tag_id')
        parent_id_str = f"{SOURCE_PREFIX}:{parent_id_raw}" if parent_id_raw else ""
        
        relative_path = get_absolute_path(node_id, nodes_dict)
        
        # --- PATHING RULE: Fix Doubled Roots ---
        if relative_path.startswith(f"{group_name} > "):
            hierarchy_path = relative_path
        elif relative_path == group_name:
            hierarchy_path = relative_path
        elif group_name == "Religious Text" and relative_path.startswith("Text > "):
            hierarchy_path = relative_path.replace("Text > ", "Religious Text > ", 1)
        elif group_name == "Religious Text" and relative_path == "Text":
            hierarchy_path = "Religious Text"
        else:
            hierarchy_path = f"{group_name} > {relative_path}"
        
        extracted_data = {
            'Source_System': SOURCE_NAME,
            'Primary_Label': name,
            'CURIE': f"{SOURCE_PREFIX}:{cid}",
            'Formal_Label': "",
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': hierarchy_path,
            'Synonyms': "",
            'Description': "",  
            'Parent_IDs': parent_id_str, 
            'Concept_ID': cid,
            'URI': "",  
            'Status': "active",
            'Crosswalks': ""
        }
        
        all_rows.append(finalize_row(extracted_data))

# --- 6. Export to Bronze Layer ---
if all_rows:
    export_df = pd.DataFrame(all_rows)[COLUMN_ORDER]
    export_df.to_csv(raw_ingest_file, index=False, encoding='utf-8-sig')
    print(f"\n[COMPLETE] Successfully wrote {len(all_rows)} DRH taxonomy concepts to {os.path.basename(raw_ingest_file)}")
else:
    print("\n[!] Warning: No valid data rows were parsed from the DRH response.")

In [ ]:
# ==============================================================================
# CELL 2 / PHASE 1: DATABASE OF RELIGIOUS HISTORY (DRH) - POLL VARIABLES
# 
# ARCHITECTURE NOTE: The DRH "Polls" are structured sociological survey 
# instruments. This script executes a GraphQL query to fetch the variables for 
# Poll 43 ("Religious Group v6"), including branching logic (parent_question_id).
# 
# SSSOM ALIGNMENT STRATEGY: 
# Targets only the 'Beliefs' and 'Practices' categories.
# Synthesizes namespaced CURIEs to prevent ID collisions with Tagging Trees:
#   - Categories -> DRH:PC[id]
#   - Groups     -> DRH:PG[id]
#   - Questions  -> DRH:P[id]
#
# BRANCHING LOGIC RULE: Survey sub-questions (e.g., conditional follow-ups) 
# are intentionally excluded from becoming their own standalone rows. Instead, 
# they are bundled and appended to the `Description` of their parent question.
# ==============================================================================

import os
import sys
import requests
import re
import pandas as pd
import time
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "DRH"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="Database of Religious History",
    fallback_uri="" 
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}_polls.csv")

GRAPHQL_URL = "https://religiondatabase.org/v1/graphql"
HEADERS = {
    'User-Agent': 'ReligiousMappingProject/1.0',
    'Content-Type': 'application/json'
}

# Updated Query to pull parent_question_id and native descriptions
GRAPHQL_QUERY = """
query GetPollStructure {
  polls_poll_by_pk(id: 43) {
    name
    categories {
      id
      name
      questions {
        id
        name
        description
        parent_question_id
      }
      groups {
        id
        name
        questions {
          id
          name
          description
          parent_question_id
        }
      }
    }
  }
}
"""

TARGET_CATEGORIES = ["Beliefs", "Practices"]

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
    else:
        print(f"[*] Found existing DRH polls file. Overwriting for fresh bulk GraphQL extraction...")
        os.remove(raw_ingest_file)

# --- 4. Text Normalization Helpers ---
def clean_text(text):
    if not text: return ""
    return re.sub(r'\s+', ' ', str(text)).strip()

def clean_label(text):
    text = clean_text(text)
    return re.sub(r'[:\-\s]+$', '', text)

# --- 5. Main Extraction & API Call ---
print(f"[*] Executing GraphQL query for Poll 43 against {GRAPHQL_URL}...")

payload = {"query": GRAPHQL_QUERY}
all_rows = []

max_retries = 3
for attempt in range(max_retries):
    try:
        response = requests.post(GRAPHQL_URL, headers=HEADERS, json=payload, timeout=20)
        response.raise_for_status()
        json_data = response.json()
        break
    except requests.exceptions.RequestException as e:
        print(f"[!] API Request failed (Attempt {attempt+1}/{max_retries}): {e}")
        if attempt == max_retries - 1:
            print("[!] CRITICAL ERROR: Maximum retries reached. Exiting.")
            sys.exit(1)
        time.sleep(2)

poll = json_data.get('data', {}).get('polls_poll_by_pk', {})
poll_name = clean_text(poll.get('name', 'Unknown Poll'))
categories = poll.get('categories', [])

print(f"[*] Successfully retrieved data for '{poll_name}'.")

# Dictionary to hold all questions temporarily for relationship mapping
questions_map = {}

# Pass 1: Build the Structural Nodes and Map the Questions
for category in categories:
    cat_name = clean_text(category.get('name', ''))
    cat_id = f"PC{category.get('id')}"
    
    if cat_name not in TARGET_CATEGORIES:
        print(f"    [-] Skipping out-of-scope category: '{cat_name}'")
        continue
        
    print(f"    [+] Processing Category: '{cat_name}'")
    
    cat_path = f"{poll_name} > {cat_name}"
    
    extracted_cat = {
        'Source_System': SOURCE_NAME,
        'Primary_Label': cat_name,
        'CURIE': f"{SOURCE_PREFIX}:{cat_id}",
        'Formal_Label': "",
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': cat_path,
        'Synonyms': "",
        'Description': "DRH Poll Category",  
        'Parent_IDs': "", 
        'Concept_ID': cat_id,
        'URI': "",  
        'Status': "active",
        'Crosswalks': ""
    }
    all_rows.append(finalize_row(extracted_cat))

    for group in category.get('groups', []):
        grp_name = clean_text(group.get('name', ''))
        grp_id = f"PG{group.get('id')}"
        grp_path = f"{cat_path} > {grp_name}"
        
        extracted_grp = {
            'Source_System': SOURCE_NAME,
            'Primary_Label': grp_name,
            'CURIE': f"{SOURCE_PREFIX}:{grp_id}",
            'Formal_Label': "",
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': grp_path,
            'Synonyms': "",
            'Description': "DRH Poll Group",  
            'Parent_IDs': f"{SOURCE_PREFIX}:{cat_id}", 
            'Concept_ID': grp_id,
            'URI': "",  
            'Status': "active",
            'Crosswalks': ""
        }
        all_rows.append(finalize_row(extracted_grp))
        
        # Collect Grouped Questions
        for q in group.get('questions', []):
            questions_map[q.get('id')] = {
                'raw_data': q,
                'path': grp_path,
                'parent_node_id': f"{SOURCE_PREFIX}:{grp_id}",
                'sub_questions': []
            }

    # Collect Flat Questions
    for q in category.get('questions', []):
        if q.get('id') not in questions_map:
            questions_map[q.get('id')] = {
                'raw_data': q,
                'path': cat_path,
                'parent_node_id': f"{SOURCE_PREFIX}:{cat_id}",
                'sub_questions': []
            }

# Pass 2: Wire up the Sub-Questions
for q_id, q_dict in questions_map.items():
    parent_id = q_dict['raw_data'].get('parent_question_id')
    if parent_id and parent_id in questions_map:
        clean_sub_name = clean_label(q_dict['raw_data'].get('name', ''))
        questions_map[parent_id]['sub_questions'].append(clean_sub_name)

# Pass 3: Export ONLY Top-Level Questions to the Bronze Layer
for q_id, q_dict in questions_map.items():
    raw_data = q_dict['raw_data']
    
    # Skip if this is a sub-question (it was already bundled into its parent)
    if raw_data.get('parent_question_id') is not None:
        continue
        
    raw_q_name = clean_text(raw_data.get('name', ''))
    clean_q_name = clean_label(raw_q_name)
    q_curie_id = f"P{q_id}"
    q_path = f"{q_dict['path']} > {clean_q_name}"
    
    # Build a rich description combining ONLY the native description and sub-questions
    desc_parts = []
    native_desc = clean_text(raw_data.get('description'))
    if native_desc:
        desc_parts.append(f"Note: {native_desc}")
    if q_dict['sub_questions']:
        sub_str = "; ".join(q_dict['sub_questions'])
        desc_parts.append(f"Conditional Sub-Questions: {sub_str}")
        
    final_desc = " | ".join(desc_parts)

    extracted_q = {
        'Source_System': SOURCE_NAME,
        'Primary_Label': clean_q_name,
        'CURIE': f"{SOURCE_PREFIX}:{q_curie_id}",
        'Formal_Label': "",
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': q_path,
        'Synonyms': "",
        'Description': final_desc,
        'Parent_IDs': q_dict['parent_node_id'], 
        'Concept_ID': q_curie_id,
        'URI': "",  
        'Status': "active",
        'Crosswalks': ""
    }
    all_rows.append(finalize_row(extracted_q))

# --- 6. Export to Bronze Layer ---
if all_rows:
    export_df = pd.DataFrame(all_rows)[COLUMN_ORDER]
    export_df.to_csv(raw_ingest_file, index=False, encoding='utf-8-sig')
    print(f"\n[COMPLETE] Successfully wrote {len(all_rows)} DRH Poll variables to {os.path.basename(raw_ingest_file)}")
else:
    print("\n[!] Warning: No valid data rows were parsed.")

In [ ]:
import requests

GRAPHQL_URL = "https://religiondatabase.org/v1/graphql"
HEADERS = {
    'User-Agent': 'ReligiousMappingProject/1.0',
    'Content-Type': 'application/json'
}

# Standard GraphQL introspection query to reveal the schema of a specific table
GRAPHQL_QUERY = """
query {
  __type(name: "polls_question") {
    fields {
      name
    }
  }
}
"""

print("[*] Fetching DRH 'polls_question' schema...")
response = requests.post(GRAPHQL_URL, headers=HEADERS, json={"query": GRAPHQL_QUERY})

if response.status_code == 200:
    data = response.json()
    fields = data.get('data', {}).get('__type', {}).get('fields', [])
    if fields:
        print("\nAvailable fields on a Question:")
        for f in fields:
            print(f" - {f['name']}")
    else:
        print("[!] No fields found. The type name might be different.")
else:
    print(f"[!] API Error: {response.status_code}\n{response.text}")
    